# Vietnamese Receipt OCR — Train (Qwen3-VL-2B + Unsloth + LoRA)

In [ ]:
# Cell 1: install package
import subprocess
import sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "git+https://github.com/<your-username>/vietnamese-receipt-ocr.git@main",
])

In [ ]:
# Cell 2: verify GPU
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3: secrets — already set via Kaggle "Add-ons → Secrets"
from kaggle_secrets import UserSecretsClient
us = UserSecretsClient()
import os
os.environ["WANDB_API_KEY"] = us.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = us.get_secret("HF_TOKEN")
print("secrets set")

In [ ]:
# Cell 4: dataset path — Kaggle attached dataset
from pathlib import Path
MCOCR = Path("/kaggle/input/vietnamese-receipts-mc-ocr-2021/versions/17")
assert MCOCR.exists(), f"Attach the MC-OCR Kaggle dataset; expected at {MCOCR}"
print("dataset OK:", MCOCR)

In [ ]:
# Cell 5: train
# Full receipt JPGs for BOTH splits live in {MCOCR}/train_images/train_images/.
# The val_images/ dir under the dataset holds a different unlabeled set; ignore it for line-OCR.
import subprocess, sys
ret = subprocess.call([
    sys.executable, "-m", "vn_receipt_ocr", "train",
    "--config", "configs/experiments/baseline_v1.yaml",
    "--override", "hf_hub.repo_owner=<your-hf-username>",
    "--override", f"data.train_path={MCOCR}/text_recognition_train_data.txt",
    "--override", f"data.train_images_dir={MCOCR}/train_images/train_images",
    "--override", f"data.val_path={MCOCR}/text_recognition_val_data.txt",
    "--override", f"data.val_images_dir={MCOCR}/train_images/train_images",
])
print("exit code:", ret)

In [ ]:
# Cell 6: print W&B run URL + HF Hub repo URL
import wandb
run = wandb.api.runs("vn-receipt-ocr")[-1]
print("W&B:", run.url)
print("HF Hub: https://huggingface.co/<your-hf-username>/vn-receipt-ocr-<run_id>")